# Datasets preprocessing

In [ ]:
import pandas as pd
from pandas.io.parsers import read_csv
from sklearn.model_selection import train_test_split
from feature_engine.creation import CyclicalFeatures

## Census
https://archive.ics.uci.edu/dataset/20/census+income

In [ ]:
# read data
path = './census/'
dftrain = read_csv(path + 'original_train.csv',header = 0, sep = ',', skipinitialspace=True)
dftest = read_csv(path + 'original_test.csv',header = 0, sep = ',', skipinitialspace=True)

trainnum = len(dftrain)
df = pd.concat([dftrain, dftest])

# Change class <=50k and >50k to 0 and 1
df["class"] = ((df["class"] == ">50K") | (df["class"] == ">50K.")).astype(int)
df.rename(columns={"class": "Y"}, inplace=True)

# Binarize categories
df = pd.get_dummies(df, columns=df.select_dtypes(include=["object", "category"]).columns, dtype=int)
dftrain = df[:trainnum]
dftest = df[trainnum:]

# Save as csv
dftrain.to_csv(path + "train.csv", index=False)
dftest.to_csv(path + "test.csv", index=False)


## Coupon
https://archive.ics.uci.edu/dataset/603/in+vehicle+coupon+recommendation

In [ ]:
# read data
path = './coupon/'
df = read_csv(path + 'in-vehicle-coupon-recommendation.csv',header = 0, sep = ',', skipinitialspace=True)

# Binarize categories
df = pd.get_dummies(df, columns=df.select_dtypes(include=["object", "category"]).columns, dtype=int)

# Split into test and train sets
dftrain, dftest = train_test_split(df, train_size=0.8, random_state=2026, stratify=df["Y"])

# Save as csv
dftrain.to_csv(path + "train.csv", index=False)
dftest.to_csv(path + "test.csv", index=False)

## stop-and-frisk
https://www.nyc.gov/site/nypd/stats/reports-analysis/stopfrisk.page

I manually concatnated the 2014, 2015 and 2016 csv by hand (copypasting) for the thesis

In [ ]:
# read data
path = './stop&frisk/'
df = read_csv(path + 'sqf-2014-2016.csv',header = 0, sep = ',', skipinitialspace=True, low_memory=False)
df.dropna(axis=0, how="all", inplace=True)
df.dropna(axis=1, how="all", inplace=True)
df2014 = df[:27226]

# Remove ** in two columns
df = df[df["age"] != "**"]
df = df[df["perstop"] != "**"]
df["age"] = df["age"].astype(float)

# Change truth letters to numbers
yn_cols = [
    c for c in df.columns
    if set(df2014[c].dropna().astype(str).str.strip().str.upper().unique()) <= {"Y", "N", "I", "O", "S", "V"}
]
df[yn_cols] = df[yn_cols].fillna(0)
df[yn_cols] = df[yn_cols].apply(
    lambda s: s.astype(str).str.strip().str.upper().map({"Y": 1, "N": 0, "1": 1, "0": 0, "I": 1, "O": 0, "S": 1, "V": 1})
).astype(int)

# It is about predicting if "someone" will get frisked, so choose features relevant to that
df = df[["datestop", "timestop", "inout", "trhsloc", "sex","race","age","ht_feet","ht_inch","weight","haircolr","eyecolor","build","addrtyp","city","sector","beat","xcoord","ycoord", "frisked"]]

# Handle stop times and dates
df["datestop"] = pd.to_datetime(df["datestop"].astype(str), format="%m%d%Y")
df["timestop"] = pd.to_datetime(df["timestop"].astype(str), format="%H%M", errors="coerce")
# Drop rows happened with errors="coerce" (Why are there "1" in "TIME OF STOP (HH:MM)" column??)
df.dropna(subset=["timestop"], inplace=True)
# Since sklearn cannot process dates, convert them into cyclical features
# https://stats.stackexchange.com/questions/311494/best-practice-for-encoding-datetime-in-machine-learning
df["datestop"] = df["datestop"].astype(int)
df["timestop"] = df["timestop"].astype(int)
cyclical = CyclicalFeatures(variables=None, drop_original=True)
dftimes = cyclical.fit_transform(df[["datestop", "timestop"]])
df = pd.concat(objs=[df, dftimes], axis=1)
df.drop(["timestop", "datestop"], axis=1, inplace=True)

# Drop object column with too many or too little different values
droplist = [i for i in df.columns if df[i].dtype == "object" and (df[i].nunique(dropna=False) > 50 or df[i].nunique(dropna=False) <= 1)]
df.drop(droplist, axis=1, inplace=True)


# Binarize categories
df = pd.get_dummies(df, columns=df.select_dtypes(include=["object", "category"]).columns, dtype=int, dummy_na=True)
df.dropna(axis=0, how="any", inplace=True)

# Rename frisked to Y
df.rename(columns={"frisked": "Y"}, inplace=True)



# Split into test and train sets
dftrain, dftest = train_test_split(df, train_size=0.8, random_state=2026, stratify=df["Y"])

# Save as csv
dftrain.to_csv(path + "train.csv", index=False)
dftest.to_csv(path + "test.csv", index=False)